In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('./data/balanced.csv')

# Construir timestamp
df['timestamp'] = (
    pd.to_datetime('2024' + df['DE13_local_date'].astype(str).str.zfill(4), format='%Y%m%d')
    + pd.to_timedelta(df['hour_local'], unit='h')
)

# Ordenar por cliente y tiempo
df = df.sort_values(['client_id', 'timestamp']).reset_index(drop=True)

# df_rolling: version con timestamp como indice, para las ventanas deslizantes
df_rolling = df.set_index('timestamp')

In [2]:
# 1. time_since_last_txn_min
# Minutos desde la ultima transaccion del cliente. Velocidad anomala = posible fraude.
df['time_since_last_txn_min'] = (
    df.groupby('client_id')['timestamp']
    .diff()
    .dt.total_seconds()
    .div(60)
    .fillna(9999)
)

In [3]:
# 2. txn_count_last_1h
# Cantidad de transacciones del cliente en la ultima hora.
df['txn_count_last_1h'] = (
    df_rolling.groupby('client_id')['DE4_amount_transaction']
    .transform(lambda x: x.rolling('1h', closed='left').count())
    .values
)

In [4]:
# 3. txn_count_last_24h
# Cantidad de transacciones del cliente en las ultimas 24 horas.
df['txn_count_last_24h'] = (
    df_rolling.groupby('client_id')['DE4_amount_transaction']
    .transform(lambda x: x.rolling('24h', closed='left').count())
    .values
)

In [5]:
# 4. amount_sum_last_1h
# Monto total gastado por el cliente en la ultima hora.
# Muchas transacciones pequeñas rapidas es patron de fraude.
df['amount_sum_last_1h'] = (
    df_rolling.groupby('client_id')['amount_usd']
    .transform(lambda x: x.rolling('1h', closed='left').sum())
    .values
)

In [6]:
# 5. amount_sum_last_24h
# Monto total gastado por el cliente en las ultimas 24 horas.
df['amount_sum_last_24h'] = (
    df_rolling.groupby('client_id')['amount_usd']
    .transform(lambda x: x.rolling('24h', closed='left').sum())
    .values
)

In [7]:
# 6. amount_zscore_customer
# Z-score del monto respecto al historico del cliente. Alta desviacion = anomalo.
df['amount_zscore_customer'] = (
    df.groupby('client_id')['amount_usd']
    .transform(lambda x: (x - x.expanding().mean()) / (x.expanding().std() + 1e-8))
)

In [8]:
# 7. amount_ratio_vs_baseline
# Ratio entre el monto de la transaccion y el monto base del cliente.
# Un ratio muy alto indica gasto inusual.
df['amount_ratio_vs_baseline'] = df['amount_usd'] / (df['client_baseline_amount'] + 1e-8)

In [9]:
# 8. is_night
# Indica si la transaccion ocurrio entre medianoche y las 5am.
# Las transacciones nocturnas tienen mayor tasa de fraude.
df['is_night'] = df['hour_local'].between(0, 5).astype(int)

In [10]:
# 9. is_high_risk_channel
# Indica si el canal es ECOM (comercio electronico), canal de mayor riesgo.
df['is_high_risk_channel'] = (df['channel'] == 'ECOM').astype(int)

In [11]:
# 10. distance_x_amount
# Interaccion entre distancia del hogar y monto.
# Compra lejos + monto alto = senal fuerte de fraude.
df['distance_x_amount'] = df['distance_from_home_km'] * df['amount_usd']

In [12]:
# Verificacion
new_features = [
    'time_since_last_txn_min', 'txn_count_last_1h', 'txn_count_last_24h',
    'amount_sum_last_1h', 'amount_sum_last_24h', 'amount_zscore_customer',
    'amount_ratio_vs_baseline', 'is_night', 'is_high_risk_channel', 'distance_x_amount'
]

print(df[new_features].describe())
print("\nPromedio por clase:")
print(df[new_features + ['is_fraud']].groupby('is_fraud').mean())

       time_since_last_txn_min  txn_count_last_1h  txn_count_last_24h  \
count             19676.000000         479.000000         1778.000000   
mean              30699.997611           1.394572            1.827334   
std               38010.598427           0.889767            1.076163   
min                   0.000000           1.000000            1.000000   
25%                7620.000000           1.000000            1.000000   
50%               12030.000000           1.000000            1.000000   
75%               43740.000000           1.000000            2.000000   
max              508800.000000           6.000000            8.000000   

       amount_sum_last_1h  amount_sum_last_24h  amount_zscore_customer  \
count          479.000000          1778.000000            15729.000000   
mean          1264.077474          1922.109724                0.111881   
std           1293.532938          2234.350064                0.897277   
min              0.070000             0.070000

In [13]:
# Guardar resultado
df.to_csv('./data/generacion_vars.csv', index=False)
print("Archivo guardado como generacion_vars.csv")

Archivo guardado como generacion_vars.csv
